In [39]:
import pandas as pd
orders = pd.read_csv('data/orders_clean.csv')
orders.shape

(800, 10)

In [40]:
orders.dtypes

order_id           str
customer_id        str
order_date         str
product            str
category           str
unit_price     float64
quantity         int64
express           bool
coupon_code        str
total          float64
dtype: object

When saving file after saving data types are not saved with it. Ex. Date staored as str

In [41]:
orders = pd.read_csv('data/orders_clean.csv',parse_dates=['order_date'])
print(orders['order_date'].dtype)

datetime64[us]


Missing values in the orders: Check null columns for all columns

In [42]:
orders.isna().sum()

order_id         0
customer_id      0
order_date       4
product          0
category         0
unit_price       0
quantity         0
express          0
coupon_code    635
total            0
dtype: int64

In [43]:
(orders.isna().mean() * 100).round(1)

order_id        0.0
customer_id     0.0
order_date      0.5
product         0.0
category        0.0
unit_price      0.0
quantity        0.0
express         0.0
coupon_code    79.4
total           0.0
dtype: float64

Show the rows where orders are missing

In [44]:
orders[orders['order_date'].isna()]

,order_id,customer_id,order_date,product,category,unit_price,quantity,express,coupon_code,total
47,ORD-20048,C0082,NaT,T-shirt,Clothing,199.0,1,True,NaN,199.0
95,ORD-20096,C0107,NaT,Yoga mat,Sports,249.0,1,False,FRIEND15,249.0
522,ORD-20523,C0075,NaT,Yoga mat,Sports,249.0,2,True,NaN,498.0
763,ORD-20764,C0030,NaT,Dumbbell set,Sports,599.0,3,False,SUMMER20,1797.0


How to access is it normal for missing column values?
1. Missing for a reason 
2. Missing by accident

In [45]:
orders['used_coupon'] = orders['coupon_code'].notna()
orders['used_coupon'].value_counts()

used_coupon
False    635
True     165
Name: count, dtype: int64

New table to read : customers

In [46]:
customers_raw = pd.read_csv('data/customers.csv')
customers = customers_raw.copy()

customers.shape

(297, 3)

In [47]:
customers.head(8)

,customer_id,city,age
0,C0001,Göteborg,41.0
1,C0002,Västerås,999.0
2,C0003,Stockholm,38.0
3,C0004,Västerås,54.0
4,C0005,Göteborg,31.0
5,C0006,NaN,40.0
6,C0007,Stockholm,39.0
7,C0008,Stockholm,50.0


Note: When some of the fields missing values pandas think datatype as float

In [48]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  297 non-null    str    
 1   city         277 non-null    str    
 2   age          267 non-null    float64
dtypes: float64(1), str(2)
memory usage: 7.1 KB


Few high age values pulls average value higher

In [49]:
customers['age'].describe()

count    267.000000
mean      69.947566
std      174.235314
min        0.000000
25%       30.000000
50%       38.000000
75%       46.500000
max      999.000000
Name: age, dtype: float64

In [50]:
print((customers['age'] == 999).sum())
print((customers['age'] == 0).sum())

9
3


In [51]:
print(customers.duplicated().sum())
print(customers['customer_id'].duplicated().sum())

5
5


Identified probems n customers:
1. **20 cities missing**
2. **30 ages are missing**
3. **9 ages are 999, a placeholder**
4. **3 ages are 0, which is impossible**

## what so we do with missing values:
choices
1. **Remove columns with missing values**
2. **Fill in missing values with something**
3. **Leave them as they are**

For age we defenitely do fix since it affect calculation:
solution: age should be 15 and 99 reasanale for people purchasing online. ITs our own judgement not written anywhere

In [53]:
customers['age'] = customers['age'].where(customers['age'].between(15,100))

print(customers['age'].isna().sum())

42


Check if the problem with mean resolved after solving placeholder:

In [54]:
customers['age'].describe()

count    255.000000
mean      37.980392
std       11.172397
min       16.000000
25%       30.000000
50%       38.000000
75%       45.000000
max       70.000000
Name: age, dtype: float64

Removing rows : dropna()

In [58]:
print(customers.dropna().shape)# any column
print(customers.dropna(subset=['city']).shape)# city is missing
print(customers.dropna(subset=['age']).shape)# age is missing

(235, 3)
(277, 3)
(255, 3)


Filling in: fillna()

In [59]:
customers['city'] = customers['city'].fillna('unknown')
customers['city'].value_counts()

city
Stockholm    88
Göteborg     50
Malmö        35
Uppsala      33
Linköping    21
unknown      20
Västerås     19
Örebro       19
Umeå         12
Name: count, dtype: int64

class 1 -> avg = 38 everybody between 36 and 40
class 1 -> avg = 38 everybody between 20 and 60

how spread out the ages are? standard deviation
in pandas : 
std bigger std - values spread out
smaller std - values are closer to mean (similar values)

In [65]:
age_median = customers['age'].median()
age_filled = customers['age'].fillna(age_median)
print(age_median)
print(customers['age'].mean().round(2),age_filled.mean().round(2))
print(customers['age'].std().round(2),age_filled.std().round(2))

38.0
37.98 37.98
11.17 10.35


Summarize 3 choices:
1. **Remove** (drona): simple, but you lose the reset of the row.use subset to be careful
2. **Fill in** (fillna): good when there is a natural value. like `unknown`. Be carful with numbers. Filled in values is a guess, and it changes the spread
3. **leave it**: often the best for numbers. Pandas skips missing values in calculations

## Merge: putting two tables together

In [66]:
orders_small = pd.DataFrame({
 'order_id': ['A1','A2','A3'],
 'customer_id': ['C1','C2','C9']
})

customers_small = pd.DataFrame({
    'customer_id': ['C1','C2','C3'],
    'city': ['Stockholm','Malma','Uppsala']
})

orders_small.merge(customers_small,on='customer_id',how='left')

,order_id,customer_id,city
0,A1,C1,Stockholm
1,A2,C2,Malma
2,A3,C9,NaN


In [67]:
orders_small.merge(customers_small,on='customer_id',how='inner')

,order_id,customer_id,city
0,A1,C1,Stockholm
1,A2,C2,Malma


Every order should get customer name and city

In [68]:
print(orders.shape)
print(customers.shape)

(800, 11)
(297, 3)


In [69]:
merged = orders.merge(customers,on='customer_id',how='left')

customers appeared for more than once mapped multiple times

In [70]:
print(merged.shape)

(813, 13)


In [71]:
customers[customers['customer_id'].duplicated(keep=False)].sort_values('customer_id')

,customer_id,city,age
8,C0009,Linköping,53.0
9,C0009,Linköping,53.0
10,C0010,Uppsala,47.0
11,C0010,Uppsala,47.0
69,C0070,Stockholm,38.0
70,C0070,Stockholm,38.0
97,C0097,Linköping,35.0
98,C0097,Linköping,35.0
197,C0199,Malmö,31.0
198,C0199,Malmö,31.0


In [73]:
print(merged['order_id'].duplicated().sum())
print(merged['total'].sum().round(1))
print(orders['total'].sum().round(1))
print((merged['total'].sum() - orders['total'].sum()).round(1))

13
585457.0
572783.8
12673.2


In [ ]:
orders.merge(customers,on='customer_id',how='left',validate='many_to_one')

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 customer_id
      C0009
      C0010
      C0070
      C0097
      C0199 ...

Remove the duplicate rows and adjust the row numbers

In [76]:
customers = customers.drop_duplicates().reset_index()

print(customers.shape)
print(customers['customer_id'].is_unique)

(292, 4)
True


In [81]:
merged = orders.merge(customers,on='customer_id',how='left',validate='many_to_one')

print(merged.shape)
print(merged['total'].sum().round(1))
print(orders['total'].sum().round(1))

(800, 14)
572783.8
572783.8


In [82]:
customers.merge(orders,on='customer_id',how='left',validate='one_to_many')

,index,customer_id,city,age,order_id,order_date,product,category,unit_price,quantity,express,coupon_code,total,used_coupon
0,0,C0001,Göteborg,41.0,ORD-20127,2026-01-30,City sneakers,Shoes,899.0,1.0,False,NaN,899.0,False
1,0,C0001,Göteborg,41.0,ORD-20190,2026-02-12,Rain jacket,Clothing,1199.0,1.0,True,WELCOME10,1199.0,True
2,0,C0001,Göteborg,41.0,ORD-20243,2026-02-24,Phone charger,Electronics,159.2,3.0,False,NaN,477.6,False
3,0,C0001,Göteborg,41.0,ORD-20723,2026-06-09,Phone charger,Electronics,159.2,1.0,False,NaN,159.2,False
4,1,C0002,Västerås,NaN,ORD-20359,2026-03-22,Wireless earbuds,Electronics,799.0,1.0,False,NaN,799.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
789,295,C0299,Stockholm,29.0,ORD-20210,2026-02-16,Throw blanket,Home & Garden,399.0,3.0,False,NaN,1197.0,False
790,295,C0299,Stockholm,29.0,ORD-20364,2026-03-23,Trail running shoes,Shoes,1299.0,2.0,False,NaN,2598.0,False
791,295,C0299,Stockholm,29.0,ORD-20617,2026-05-16,Dumbbell set,Sports,599.0,3.0,False,NaN,1797.0,False
792,296,C0300,Stockholm,16.0,ORD-20573,2026-05-09,Fleece sweater,Clothing,699.0,1.0,False,WELCOME10,699.0,True


## What did we lose? Using the merged table
indicator = True
_merged

In [86]:
# Find out which order did NOT find a customer
merged = orders.merge(customers,on='customer_id',how='left',validate='many_to_one',indicator=True)

merged['_merge'].value_counts()
# both - the order found it customer
# left_only - The order has no customer in the customer table
# right_only A customer without orders (only with how='right' or 'outer')

_merge
both          771
left_only      29
right_only      0
Name: count, dtype: int64

In [ ]:
no_customers = merged[merged['_merge'] == 'left_only']

print(no_customers['customer_id'].nunique())
print(no_customers['total'].sum())

8
17438.6


In [92]:
merged_inner = orders.merge(customers,on='customer_id',how='right',validate='many_to_one',indicator=True)
print(merged_inner.shape)

(794, 15)


which group buys most?

In [93]:
merged['city'] = merged['city'].fillna('No customer record')
revenue_per_city = merged.groupby('city')['total'].sum().round(0).sort_values(ascending=False)

revenue_per_city

city
Stockholm             175321.0
Göteborg               92686.0
Uppsala                73317.0
Malmö                  65224.0
Örebro                 40994.0
unknown                29417.0
Västerås               28780.0
Linköping              25058.0
Umeå                   24548.0
No customer record     17439.0
Name: total, dtype: float64

What is the average age of customers in each city?
Hint: .agg

In [95]:
customers.groupby('city')['age'].agg(['count','mean']).round(1)

,count,mean
city,,
Göteborg,43,37.3
Linköping,19,43.1
Malmö,24,38.0
Stockholm,76,37.2
Umeå,11,40.6
Uppsala,29,41.6
Västerås,15,37.2
unknown,20,35.2
Örebro,13,31.2
